In [1]:
from sqlalchemy import select, or_, between, func, desc
from models import Product
from db import Session

In [2]:
session = Session()

In [3]:
first_three = select(Product).where(Product.year== 1983).order_by(Product.name).limit(3)

session.execute(first_three).all()


2026-09-04 14:47:28,449 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 14:47:28,455 INFO sqlalchemy.engine.Engine SELECT products.id, products.name, products.manufacturer_id, products.year, products.country, products.cpu 
FROM products 
WHERE products.year = ? ORDER BY products.name
 LIMIT ? OFFSET ?
2026-09-04 14:47:28,455 INFO sqlalchemy.engine.Engine [generated in 0.00075s] (1983, 3, 0)


[(Product(17, "Apple IIe"),),
 (Product(85, "Aquarius"),),
 (Product(38, "Coleco Adam"),)]

In [4]:
z80_cpu = (select(Product).where(Product.cpu.like('%Z80%')))
session.scalars(z80_cpu).all()

2026-09-04 14:47:28,479 INFO sqlalchemy.engine.Engine SELECT products.id, products.name, products.manufacturer_id, products.year, products.country, products.cpu 
FROM products 
WHERE products.cpu LIKE ?
2026-09-04 14:47:28,480 INFO sqlalchemy.engine.Engine [generated in 0.00076s] ('%Z80%',)


[Product(7, "CPC 464"),
 Product(8, "CPC 664"),
 Product(9, "CPC 6128"),
 Product(10, "464 Plus"),
 Product(11, "6128 Plus"),
 Product(12, "PCW"),
 Product(23, "CT-80"),
 Product(34, "Bally Brain"),
 Product(35, "Bally Astrocade"),
 Product(36, "CoBra"),
 Product(37, "Lynx"),
 Product(38, "Coleco Adam"),
 Product(47, "Commodore 128"),
 Product(51, "VTech Laser 200"),
 Product(52, "Video Genie"),
 Product(53, "Colour Genie"),
 Product(54, "Rabbit RX83"),
 Product(55, "Alpha"),
 Product(56, "Beta"),
 Product(57, "Gama"),
 Product(60, "Dubna 48K"),
 Product(65, "Exidy Sorcerer"),
 Product(66, "Enterprise 64"),
 Product(67, "Enterprise 128"),
 Product(68, "Lambda 8300"),
 Product(72, "Grundy NewBrain"),
 Product(73, "Gradiente Expert"),
 Product(79, "Hobbit"),
 Product(83, "Jupiter ACE"),
 Product(84, "ABC 80"),
 Product(85, "Aquarius"),
 Product(87, "MTX500"),
 Product(88, "MTX512"),
 Product(89, "RS128"),
 Product(90, "MicroBee"),
 Product(91, "CCE MC-1000"),
 Product(92, "TK82C"),
 Prod

In [5]:
products = select(Product).where(or_(Product.cpu.like('%Z80%'), Product.cpu.like('%6502%'))).where(Product.year<1990).order_by(Product.name)
session.execute(products).all()

2026-09-04 14:47:28,507 INFO sqlalchemy.engine.Engine SELECT products.id, products.name, products.manufacturer_id, products.year, products.country, products.cpu 
FROM products 
WHERE (products.cpu LIKE ? OR products.cpu LIKE ?) AND products.year < ? ORDER BY products.name
2026-09-04 14:47:28,508 INFO sqlalchemy.engine.Engine [generated in 0.00090s] ('%Z80%', '%6502%', 1990)


[(Product(84, "ABC 80"),),
 (Product(1, "Acorn Atom"),),
 (Product(55, "Alpha"),),
 (Product(16, "Apple II"),),
 (Product(20, "Apple II Plus"),),
 (Product(17, "Apple IIe"),),
 (Product(85, "Aquarius"),),
 (Product(2, "BBC Micro"),),
 (Product(35, "Bally Astrocade"),),
 (Product(34, "Bally Brain"),),
 (Product(56, "Beta"),),
 (Product(91, "CCE MC-1000"),),
 (Product(14, "CEC-I Zhonghua"),),
 (Product(7, "CPC 464"),),
 (Product(9, "CPC 6128"),),
 (Product(8, "CPC 664"),),
 (Product(23, "CT-80"),),
 (Product(36, "CoBra"),),
 (Product(38, "Coleco Adam"),),
 (Product(53, "Colour Genie"),),
 (Product(45, "Commodore 116"),),
 (Product(47, "Commodore 128"),),
 (Product(44, "Commodore 16"),),
 (Product(3, "Electron"),),
 (Product(67, "Enterprise 128"),),
 (Product(66, "Enterprise 64"),),
 (Product(65, "Exidy Sorcerer"),),
 (Product(69, "Franklin ACE"),),
 (Product(111, "G7480"),),
 (Product(149, "GEM 1000"),),
 (Product(108, "Galeb"),),
 (Product(57, "Gama"),),
 (Product(73, "Gradiente Expert"

In [6]:
manufactures = select(Product.manufacturer).where(Product.year.between(1980,1989)).distinct()
session.execute(manufactures).all()

2026-09-04 14:47:28,557 INFO sqlalchemy.engine.Engine SELECT DISTINCT manufacturers.id = products.manufacturer_id AS anon_1 
FROM manufacturers, products 
WHERE products.year BETWEEN ? AND ?
2026-09-04 14:47:28,558 INFO sqlalchemy.engine.Engine [generated in 0.00069s] (1980, 1989)


C:\Users\AuricErgesonNitonde\AppData\Local\Temp\ipykernel_10032\3847196778.py:2: SAWarning: SELECT statement has a cartesian product between FROM element(s) "manufacturers" and FROM element "products".  Apply join condition(s) between each element to resolve.
  session.execute(manufactures).all()


[(True,), (False,)]

In [7]:
manufactures_t = select(Product.manufacturer).where(Product.manufacturer.like('T%')).order_by(Product.manufacturer).distinct()
session.execute(manufactures_t).all()

NotImplementedError: <function like_op at 0x0000024A8480F100>

In [8]:
croatia = (select(
           func.min(Product.year),
           func.max(Product.year),
           func.count()
           )
            .group_by(Product.country)
            . having(Product.country=='Croatia')

           )

session.execute(croatia).first()

2026-09-04 14:47:37,142 INFO sqlalchemy.engine.Engine SELECT min(products.year) AS min_1, max(products.year) AS max_1, count(*) AS count_1 
FROM products GROUP BY products.country 
HAVING products.country = ?
2026-09-04 14:47:37,143 INFO sqlalchemy.engine.Engine [generated in 0.00065s] ('Croatia',)


(1981, 1984, 4)

In [9]:
counts = func.count()
num_products = (
    select(Product.year,
           counts,
    )
    .group_by(Product.year)
    .order_by(counts.desc())
)

session.execute(num_products).all()

2026-09-04 14:47:39,787 INFO sqlalchemy.engine.Engine SELECT products.year, count(*) AS count_1 
FROM products GROUP BY products.year ORDER BY count(*) DESC
2026-09-04 14:47:39,788 INFO sqlalchemy.engine.Engine [generated in 0.00079s] ()


[(1983, 21),
 (1984, 21),
 (1985, 19),
 (1982, 17),
 (1986, 11),
 (1980, 10),
 (1977, 7),
 (1979, 7),
 (1981, 6),
 (1987, 6),
 (1990, 5),
 (1989, 4),
 (1978, 2),
 (1988, 2),
 (1969, 1),
 (1991, 1),
 (1992, 1),
 (1995, 1)]

In [10]:
usas = select(Product.manufacturer,
            func.count(Product.manufacturer.distinct()),
           ).group_by(Product.country).having(Product.country == 'USA')



session.execute(usas).all()

NotImplementedError: <function distinct_op at 0x0000024A8480F600>

In [11]:
q = (select(func.count(Product.manufacturer.distinct()))
.where(Product.country == 'USA'))

session.scalar(q)

NotImplementedError: <function distinct_op at 0x0000024A8480F600>